# CricketIQ: IPL auction analysis (run once, offline)
This notebook cleans the past auction files, matches player names, builds each player's stats, trains a price model, and exports two files for the app:
- `players_pool.csv` (every player: role, score 0-100, fair price in crore)
- `teams_2026.csv` (last season table position, batting and bowling rank per team)

**How to run:** Runtime > Run all. In step 1 upload the 6 data files (the zips are fine).

**To refresh for a new year:** upload the new files, set `AUCTION_YEAR` in step 7 to the next auction year, run all, then replace `data.json` in the app (see README).

## Step 1. Upload the data files
Upload: `IPL2021_Auction_v2.csv`, `IPL2022_Sold_Players.csv`, `IPL Auction 2023.csv`, `IPL_2024_Players_Auction_Dataset.csv`, `ipl_2025_auction_players.csv`, `ipl-ball-by-ball-dataset.csv` (CSV or inside zips).

In [ ]:
import os, glob, shutil, zipfile
from google.colab import files
os.makedirs('/content/raw', exist_ok=True)
up = files.upload()
for n in up:
    if n.lower().endswith('.zip'):
        zipfile.ZipFile(n).extractall('/content/raw')
    else:
        shutil.move(n, '/content/raw/'+n)
want = {'a6':'IPL2021_Auction_v2.csv','a2':'IPL2022_Sold_Players.csv','a4':'IPL Auction 2023.csv','a5':'IPL_2024_Players_Auction_Dataset.csv','a1':'ipl_2025_auction_players.csv','a7':'ipl-ball-by-ball-dataset.csv'}
for k, f in want.items():
    hit = glob.glob('/content/raw/**/'+f, recursive=True)
    assert hit, 'Missing file: '+f
    os.makedirs('/content/ds/'+k, exist_ok=True); shutil.copy(hit[0], '/content/ds/'+k+'/'+f)
print('All files found.')
os.makedirs('/content/scripts', exist_ok=True); os.makedirs('/content/out', exist_ok=True)

## Helper code

In [ ]:
%%writefile /content/scripts/common.py
import re
DS = "/content/ds"
OUT = "/content/out"

def team_code(s):
    s = str(s).lower()
    rules = [("chennai", "CSK"), ("mumbai", "MI"), ("royal challengers", "RCB"), ("bangalore", "RCB"),
             ("bengaluru", "RCB"), ("kolkata", "KKR"), ("sunrisers", "SRH"), ("hyderabad", "SRH"),
             ("deccan", "DCH"), ("delhi", "DC"), ("punjab", "PBKS"), ("rajasthan", "RR"),
             ("lucknow", "LSG"), ("gujarat lions", "GL"), ("gujarat", "GT"), ("gujrat", "GT"),
             ("pune warriors", "PWI"), ("supergiant", "RPS"), ("kochi", "KTK")]
    for k, v in rules:
        if k in s:
            return v
    return {"csk": "CSK", "mi": "MI", "rcb": "RCB", "kkr": "KKR", "srh": "SRH", "sh": "SRH", "dc": "DC",
            "pbks": "PBKS", "pk": "PBKS", "rr": "RR", "lsg": "LSG", "gt": "GT"}.get(s.strip(), None)

def norm(s):
    s = re.sub(r"\(.*?\)", " ", str(s))                    # drop "(T)", "(RTM)" tags
    s = re.sub(r"[\u2010-\u2015\-]", " ", s.lower()).replace(".", " ")
    s = re.sub(r"[^a-z ]", "", s)
    return re.sub(r"\s+", " ", s).strip()


## Step 2. Clean the 5 auction files into one table

In [ ]:
%%writefile /content/scripts/step1_auctions.py
import sys; sys.path.insert(0, "/content/scripts")
import pandas as pd, numpy as np, re
from common import *

rows = []
def add(year, df, **kw):
    d = pd.DataFrame(kw); d["year"] = year; rows.append(d)

role_map = {"bat": "Batter", "batsman": "Batter", "batter": "Batter", "toporder batsman": "Batter",
            "middleorder batsman": "Batter", "ar": "All-rounder", "all rounder": "All-rounder", "allrounder": "All-rounder",
            "bowling allrounder": "All-rounder", "batting allrounder": "All-rounder", "bowl": "Bowler",
            "bowler": "Bowler", "wk": "Wicketkeeper", "wicket keeper": "Wicketkeeper", "wicketkeeper": "Wicketkeeper"}
def role(x):
    return role_map.get(re.sub(r"[-_]", " ", str(x).lower()).strip())

# 2021
a = pd.read_csv(f"{DS}/a6/IPL2021_Auction_v2.csv")
sold = a["Bought_By"].notna() & a["Bought_By"].ne("Unsold")
add(2021, a, player=a["Name"], role=a["Playing_Role"].map(role),
    nationality=np.where(a["Country"].eq("India"), "Indian", "Overseas"), age_at_auction=a["Age"],
    team=a["Bought_By"].map(lambda s: team_code(s) if isinstance(s, str) and s != "Unsold" else None),
    prev_team=a["2020_Team"].map(team_code), base_cr=a["ReservePriceInLakh"] / 100.0, price_cr=a["Buy Price"] / 1e7,
    status=np.where(sold, "sold", "unsold"), capped=a["C_U_A"])
# 2022 (sold only, price like "? 2,00,00,000")
a = pd.read_csv(f"{DS}/a2/IPL2022_Sold_Players.csv")
price = a["Sold_Price"].map(lambda s: float(re.sub(r"[^0-9]", "", str(s))) / 1e7)
add(2022, a, player=a["Player_Name"], role=a["Type"].map(role), nationality=a["Nationality"],
    team=a["Teams"].map(team_code), base_cr=np.nan, price_cr=price, status="sold")
# 2023
a = pd.read_csv(f"{DS}/a4/IPL Auction 2023.csv")
ret = a["Base Price"].astype(str).eq("Retained"); uns = a["2023 Squad"].eq("Unsold Players")
base = pd.to_numeric(a["Base Price"], errors="coerce") / 1e7
st = np.where(ret, "retained", np.where(uns, "unsold", "sold"))
add(2023, a, player=a["Players"], role=a["TYPE"].map(role), nationality=None, prev_team=a["2022 Squad"].map(team_code),
    team=a["2023 Squad"].map(lambda s: None if s == "Unsold Players" else team_code(s)),
    base_cr=base, price_cr=np.where(st == "sold", a["COST IN ₹ (CR.)"], np.nan), status=st)
# 2024
a = pd.read_csv(f"{DS}/a5/IPL_2024_Players_Auction_Dataset.csv")
bp = a["Base Price"].astype(str)
st = np.where(a["Team"].eq("Unsold"), "unsold", np.where(bp.isin(["Retained", "Traded"]), "retained", "sold"))
add(2024, a, player=a["Players"], role=a["TYPE"].map(role), nationality=None, prev_team=a["2023 Squad"].map(team_code),
    team=a["Team"].map(lambda s: None if s == "Unsold" else team_code(s)),
    base_cr=pd.to_numeric(a["Base Price"], errors="coerce") / 1e7,
    price_cr=np.where(st == "sold", a["COST IN ₹ (CR.)"], np.nan), status=st)
# 2025
a = pd.read_csv(f"{DS}/a1/ipl_2025_auction_players.csv")
has_team = a["Team"].ne("-"); retained = has_team & a["Base"].eq("-")
st = np.where(retained, "retained", np.where(has_team, "sold", "unsold"))   # TBA and Unsold both = no team = unsold
add(2025, a, player=a["Players"], role=a["Type"].map(role), nationality=None,
    team=a["Team"].map(lambda s: None if s == "-" else team_code(s)),
    base_cr=pd.to_numeric(a["Base"], errors="coerce"),
    price_cr=np.where(st == "sold", pd.to_numeric(a["Sold"], errors="coerce"), np.nan), status=st)

A = pd.concat(rows, ignore_index=True)
# documented fixes for three impossible rows (price below base price)
fix = (A.year.eq(2021) & A.player.isin(["Utkarsh Singh", "Daniel Christian"]) & A.status.eq("sold"))
A.loc[fix, "price_cr"] = A.loc[fix, "price_cr"] * 10            # a zero was missing in the source file
fix = (A.year.eq(2023) & A.player.eq("Shivam Singh") & A.status.eq("sold"))
A.loc[fix, "base_cr"] = A.loc[fix, "price_cr"]                  # base price in the file was wrong
A["player"] = A["player"].astype(str).str.strip()
A["key"] = A["player"].map(norm)
A = A[["year", "player", "key", "role", "nationality", "age_at_auction", "capped", "prev_team", "team", "base_cr", "price_cr", "status"]]
# same player listed twice in a year: keep sold, then retained, then unsold
A["_p"] = A["status"].map({"sold": 0, "retained": 1, "unsold": 2})
A = A.sort_values(["year", "key", "_p"]).drop_duplicates(["year", "key"], keep="first").drop(columns="_p")
A.to_csv(f"{OUT}/auctions_clean.csv", index=False)
print(A.groupby(["year", "status"]).size().unstack(fill_value=0))
s = A[A.status == "sold"]
print("sold price crore by year:\n", s.groupby("year")["price_cr"].describe()[["count", "min", "50%", "max"]].round(2))
print("sold with missing price:", s.price_cr.isna().sum(), " missing role:", s.role.isna().sum())
print(s.sort_values("price_cr", ascending=False).groupby("year").head(1)[["year", "player", "team", "price_cr"]].to_string(index=False))


In [ ]:
!python -I /content/scripts/step1_auctions.py

## Step 3a. Load ball-by-ball data

In [ ]:
%%writefile /content/scripts/step2_load.py
import sys; sys.path.insert(0, "/content/scripts")
import pandas as pd, numpy as np
from common import *
cols = ["match_id","date","season","event","venue","team1","team2","winner","win_by_runs","win_by_wickets","player_of_match",
        "innings","batting_team","over","ball","batter","bowler","runs_batter","runs_extras","runs_total",
        "extras_wides","extras_noballs","extras_byes","extras_legbyes","wicket_kind","wicket_player_out"]
d = pd.read_csv(f"{DS}/a7/ipl-ball-by-ball-dataset.csv", usecols=cols, engine="python")
d["date"] = pd.to_datetime(d["date"])
d["year"] = d["date"].dt.year
d.to_pickle(f"{OUT}/ball.pkl")
print(d.shape)
m = d.drop_duplicates("match_id")
print(m.event.value_counts().to_dict())
print(m.groupby("year").size().to_dict())
print(d.innings.value_counts().to_dict())
print(m[m.year==2026].sort_values("date").tail(5)[["date","team1","team2","winner","venue"]].to_string(index=False))
print(sorted(set(d.wicket_kind.dropna())))
print(d[["extras_wides","extras_noballs","extras_byes","extras_legbyes","runs_extras"]].describe().loc[["min","max","mean"]].round(3))
print(d.loc[d.extras_wides>0, ["extras_wides","runs_extras","runs_total"]].head(3))


In [ ]:
!python -I /content/scripts/step2_load.py

## Step 3b. Build player and team stats per season

In [ ]:
%%writefile /content/scripts/step2_stats.py
import sys; sys.path.insert(0, "/content/scripts")
import pandas as pd, numpy as np
from common import *
d = pd.read_pickle(f"{OUT}/ball.pkl")
d = d[d.innings <= 2].copy()                                  # leave out super overs
for c in ["extras_wides","extras_noballs","runs_batter","runs_total"]:
    d[c] = d[c].fillna(0)
d["bat_team"] = d["batting_team"].map(team_code)
d["bowl_team"] = np.where(d["batting_team"] == d["team1"], d["team2"], d["team1"])
d["bowl_team"] = d["bowl_team"].map(team_code)
d["winner_code"] = d["winner"].map(team_code)

# ---------------- batting per player per season ----------------
faced = d[d.extras_wides == 0]                                  # a wide is not a ball faced
bat = faced.groupby(["batter", "year"]).agg(balls=("ball", "size"), runs=("runs_batter", "sum"),
                                            fours=("runs_batter", lambda s: (s == 4).sum()),
                                            sixes=("runs_batter", lambda s: (s == 6).sum()))
inn = d.groupby(["batter", "year"]).match_id.nunique().rename("bat_matches")
outs = d[d.wicket_player_out.notna() & ~d.wicket_kind.isin(["retired hurt"])]
outs = outs.groupby(["wicket_player_out", "year"]).size().rename("outs"); outs.index.names = ["batter", "year"]
bat = bat.join(inn).join(outs).fillna({"outs": 0}).reset_index()
bat["team_bat"] = bat.set_index(["batter", "year"]).index.map(
    d.groupby(["batter", "year"]).bat_team.agg(lambda s: s.mode().iat[0]))

# ---------------- bowling per player per season ----------------
legal = d[(d.extras_wides == 0) & (d.extras_noballs == 0)]
d["conceded"] = d.runs_batter + d.extras_wides + d.extras_noballs
bw = legal.groupby(["bowler", "year"]).agg(legal_balls=("ball", "size"))
bw = bw.join(d.groupby(["bowler", "year"]).conceded.sum())
credited = ["bowled", "caught", "caught and bowled", "lbw", "stumped", "hit wicket"]
wk = d[d.wicket_kind.isin(credited)].groupby(["bowler", "year"]).size().rename("wickets")
bw = bw.join(wk).fillna({"wickets": 0})
ph = legal.assign(phase=pd.cut(legal.over, [-1, 5, 15, 25], labels=["pp", "mid", "death"]))
ph = ph.groupby(["bowler", "year", "phase"], observed=False).size().unstack(fill_value=0)
ph.columns = [f"balls_{c}" for c in ph.columns]
bw = bw.join(ph)
bw["bowl_matches"] = d.groupby(["bowler", "year"]).match_id.nunique()
bw["team_bowl"] = d.groupby(["bowler", "year"]).bowl_team.agg(lambda s: s.mode().iat[0])
bw = bw.reset_index().rename(columns={"bowler": "name"})
bat = bat.rename(columns={"batter": "name"})
ps = bat.merge(bw, on=["name", "year"], how="outer")
ps["team"] = ps["team_bat"].fillna(ps["team_bowl"]); ps = ps.drop(columns=["team_bat", "team_bowl"])
num = [c for c in ps.columns if c not in ("name", "year", "team")]
ps[num] = ps[num].fillna(0)
ps.to_csv(f"{OUT}/player_season.csv", index=False)
print("player-seasons", len(ps), " players", ps.name.nunique())

# ---------------- pace or spin guess (career overs pattern) ----------------
c = bw.groupby("name")[["legal_balls", "balls_pp", "balls_mid", "balls_death"]].sum()
c = c[c.legal_balls >= 60]
c["pace_share"] = (c.balls_pp + c.balls_death) / c.legal_balls
c["kind"] = np.where(c.pace_share >= 0.62, "Fast bowler", "Spin bowler")
c[["legal_balls", "pace_share", "kind"]].to_csv(f"{OUT}/bowler_kind.csv")
tests = {"JJ Bumrah": "Fast", "YS Chahal": "Spin", "Rashid Khan": "Spin", "SP Narine": "Spin", "R Ashwin": "Spin",
         "Kuldeep Yadav": "Spin", "Arshdeep Singh": "Fast", "B Kumar": "Fast", "Mohammed Shami": "Fast",
         "K Rabada": "Fast", "MA Starc": "Fast", "PJ Cummins": "Fast", "Mohammed Siraj": "Fast",
         "Harshal Patel": "Fast", "T Natarajan": "Fast", "RA Jadeja": "Spin", "AR Patel": "Spin",
         "Varun Chakaravarthy": "Spin", "PP Chawla": "Spin", "A Mishra": "Spin", "Noor Ahmad": "Spin"}
ok = 0; n = 0
for k, v in tests.items():
    if k in c.index:
        n += 1; got = c.loc[k, "kind"].split()[0]; ok += got == v
        if got != v: print("  MISS", k, v, "->", got, round(c.loc[k, "pace_share"], 2))
print(f"pace/spin check on known bowlers: {ok}/{n} right")

# ---------------- team tables (league stage only) ----------------
m = d.drop_duplicates("match_id")[["match_id", "date", "year", "team1", "team2", "winner"]].sort_values(["year", "date", "match_id"])
m["t1"] = m.team1.map(team_code); m["t2"] = m.team2.map(team_code); m["w"] = m.winner.map(team_code)
m["rank_in_year"] = m.groupby("year").cumcount(ascending=False)
m["playoff"] = np.where(m.year >= 2011, m.rank_in_year < 4, m.rank_in_year < 3)
lg = m[~m.playoff]
recs = []
for _, r in lg.iterrows():
    for t in (r.t1, r.t2):
        recs.append((r.year, t, 1, int(r.w == t), int(pd.isna(r.w))))
T = pd.DataFrame(recs, columns=["year", "team", "M", "W", "NR"]).groupby(["year", "team"]).sum().reset_index()
T["L"] = T.M - T.W - T.NR; T["pts"] = 2 * T.W + T.NR
# net run rate (approximate): all out counts as 20 overs
lgid = set(lg.match_id)
x = d[d.match_id.isin(lgid)]
gi = x.groupby(["match_id", "innings", "bat_team", "bowl_team", "year"]).agg(
    runs=("runs_total", "sum"), lb=("ball", lambda s: 0)).reset_index()
lb = x[(x.extras_wides == 0) & (x.extras_noballs == 0)].groupby(["match_id", "innings"]).size().rename("legal")
wk_ = x[x.wicket_player_out.notna() & ~x.wicket_kind.isin(["retired hurt"])].groupby(["match_id", "innings"]).size().rename("wk")
gi = gi.merge(lb, on=["match_id", "innings"], how="left").merge(wk_, on=["match_id", "innings"], how="left").fillna({"wk": 0})
gi["overs"] = np.where(gi.wk >= 10, 20.0, gi.legal / 6.0)
f = gi.groupby(["year", "bat_team"]).agg(rf=("runs", "sum"), of=("overs", "sum")).rename_axis(["year", "team"])
a = gi.groupby(["year", "bowl_team"]).agg(ra=("runs", "sum"), oa=("overs", "sum")).rename_axis(["year", "team"])
T = T.merge(f.reset_index(), on=["year", "team"]).merge(a.reset_index(), on=["year", "team"])
T["nrr"] = (T.rf / T.of - T.ra / T.oa).round(3)
T["bat_rpo"] = (T.rf / T.of).round(2); T["bowl_rpo"] = (T.ra / T.oa).round(2)
T = T.sort_values(["year", "pts", "nrr"], ascending=[True, False, False])
T["pos"] = T.groupby("year").cumcount() + 1
T.to_csv(f"{OUT}/team_season.csv", index=False)
for y in (2024, 2025, 2026):
    print(y, T[T.year == y][["pos", "team", "M", "W", "L", "NR", "pts", "nrr"]].head(10).to_string(index=False))


In [ ]:
!python -I /content/scripts/step2_stats.py

## Step 4. Match auction names to stats names

In [ ]:
%%writefile /content/scripts/step3_match.py
import sys; sys.path.insert(0, "/content/scripts")
import pandas as pd, numpy as np, difflib
from common import *
A = pd.read_csv(f"{OUT}/auctions_clean.csv")
ps = pd.read_csv(f"{OUT}/player_season.csv")
tot = ps.groupby("name").agg(bat=("balls", "sum"), bowl=("legal_balls", "sum"), first=("year", "min"), last=("year", "max"))
tot["weight"] = tot.bat + tot.bowl
teams_by = ps.groupby(["name", "year"]).team.agg(lambda s: set(s.dropna())).to_dict()
cs_names = list(tot.index)
cs_tok = {n: norm(n).split() for n in cs_names}
cs_norm = {norm(n): n for n in cs_names}

# a few names the rules cannot solve (stats name is quite different from auction name); checked by hand
manual = {norm("Mohammad Shami"): "Mohammed Shami", norm("Syed Khaleel Ahmed"): "KK Ahmed", norm("Ripal Patel"): None,
          norm("Mujeeb Rahman"): "Mujeeb Ur Rahman", norm("Chris Woakes"): "CR Woakes", norm("Alex Hales"): "AD Hales",
          norm("Saurav Chuahan"): "Saurav Chauhan", norm("Shahbaz Ahamad"): "Shahbaz Ahmed",
          norm("Wanindu Hasaranga"): "PWH de Silva", norm("Varun Chakravarthy"): "CV Varun",
          norm("Varun Chakaravarthy"): "CV Varun", norm("Prasidh Krishna"): "M Prasidh Krishna",
          norm("Rinku Singh"): "RK Singh"}

def candidates(key):
    t = key.split()
    if not t: return []
    first, last = t[0], t[-1]
    out = []
    for n, ct in cs_tok.items():
        if not ct or ct[-1] != last: continue
        initials = ct[0] if (len(ct[0]) <= 3 and n.split()[0].isupper()) else ""
        given = [x for x in t[:-1] if len(x) > 2]
        if first in ct[:-1] or any(x in ct[:-1] for x in given) or (initials and first[0] in initials) \
                or (not initials and len(first) <= 2 and ct[0][:1] == first[:1]):
            out.append(n)
    return out

def plausible(role, n):
    bat, bowl = tot.loc[n, "bat"], tot.loc[n, "bowl"]
    if role == "Bowler": return bowl >= 12
    if role in ("Batter", "Wicketkeeper"): return bat >= 12
    return bat >= 12 or bowl >= 12

res = []
for key, g in A.groupby("key"):
    name = g.player.iloc[0]
    roles = [r for r in g.role.dropna()] or [None]
    role = roles[0]
    if key in manual:
        res.append((key, name, manual[key], "manual" if manual[key] else "none", 1)); continue
    if key in cs_norm:
        res.append((key, name, cs_norm[key], "exact", 1)); continue
    cands = candidates(key)
    y0 = g.year.min()
    scored = []
    for c in cands:
        if not plausible(role, c): continue
        active = tot.loc[c, "last"] >= y0 - 3
        agree = False
        for _, r in g.iterrows():
            tset = teams_by.get((c, r.year), set()) | teams_by.get((c, r.year - 1), set())
            if isinstance(r.team, str) and r.team in tset: agree = True
            if "prev_team" in g and isinstance(r.prev_team, str) and r.prev_team in teams_by.get((c, r.year - 1), set()): agree = True
        scored.append((agree, active, tot.loc[c, "weight"], c))
    scored = [s for s in scored if s[0] or s[1]]
    if scored:
        scored.sort(reverse=True); best = scored[0]
        if best[0]:
            res.append((key, name, best[3], "team+surname", len(scored))); continue
        has_given = len(norm(best[3]).split()[0]) > 3          # stats name holds a real first name, not just initials
        if len(scored) == 1 and has_given:
            res.append((key, name, best[3], "initial+surname", 1)); continue
        res.append((key, name, None, "skipped-unsure", len(scored))); continue
    t = key.split()
    close = [c for c in cs_norm if c.split()[0] == t[0] and difflib.SequenceMatcher(None, c.split()[-1], t[-1]).ratio() >= 0.85]
    if len(t) > 1 and len(close) == 1 and plausible(role, cs_norm[close[0]]):
        res.append((key, name, cs_norm[close[0]], "fuzzy", 1)); continue
    res.append((key, name, None, "none", 0))
M = pd.DataFrame(res, columns=["key", "auction_name", "cs_name", "method", "n_candidates"])
M.to_csv(f"{OUT}/name_map.csv", index=False)
print(M.method.value_counts().to_dict())
A2 = A.merge(M[["key", "cs_name", "method"]], on="key", how="left"); A2["matched"] = A2.cs_name.notna()
sr = A2[A2.status.isin(["sold", "retained"])]
print("sold+retained matched share by year:", sr.groupby("year").matched.mean().round(2).to_dict())
print("\nFUZZY:"); print(M[M.method == "fuzzy"][["auction_name", "cs_name"]].to_string(index=False))
print("\nUNMATCHED sold/retained with price >= 2 cr (or retained):")
u = sr[~sr.matched & ((sr.price_cr >= 2) | (sr.status == "retained"))][["year", "player", "status", "price_cr"]]
print(u.to_string(index=False))


In [ ]:
!python -I /content/scripts/step3_match.py

## Step 5. One row per player: role, nationality, age

In [ ]:
%%writefile /content/scripts/step4_players.py
import sys; sys.path.insert(0, "/content/scripts")
import pandas as pd, numpy as np
from common import *
A = pd.read_csv(f"{OUT}/auctions_clean.csv"); M = pd.read_csv(f"{OUT}/name_map.csv")
ps = pd.read_csv(f"{OUT}/player_season.csv"); bk = pd.read_csv(f"{OUT}/bowler_kind.csv")
bk["kind"] = np.where(bk.pace_share >= 0.50, "Fast bowler", "Spin bowler")
bk.loc[bk.name.isin(["HV Patel", "M Theekshana"]), "kind"] = bk.name.map({"HV Patel": "Fast bowler", "M Theekshana": "Spin bowler"})
bk = bk.set_index("name")
A = A.merge(M[["key", "cs_name"]], on="key", how="left")

# one row per person: auction name (best display name) + stats name
rows = []
for key, g in A.groupby("key"):
    g = g.sort_values("year")
    r = {"key": key, "name": g.player.iloc[-1], "cs_name": g.cs_name.iloc[0]}
    roles = list(g.role.dropna())
    r["role_raw"] = "Wicketkeeper" if "Wicketkeeper" in roles else ("All-rounder" if "All-rounder" in roles else
                    ("Bowler" if "Bowler" in roles else ("Batter" if "Batter" in roles else None)))
    nat = g.nationality.dropna()
    r["nationality"] = nat.iloc[-1] if len(nat) else None
    age = g.dropna(subset=["age_at_auction"])
    r["age_now"] = int(age.age_at_auction.iloc[-1] + (2026 - int(age.year.iloc[-1])) + 1) if len(age) else None
    r["last_auction_year"] = int(g.year.max())
    sold = g[g.status == "sold"]
    r["last_price_cr"] = sold.price_cr.iloc[-1] if len(sold) else None
    r["last_base_cr"] = g.base_cr.dropna().iloc[-1] if g.base_cr.notna().any() else None
    r["last_team"] = g.team.dropna().iloc[-1] if g.team.notna().any() else None
    rows.append(r)
P = pd.DataFrame(rows)
# players in the stats data with no auction row: keep them too (they played recently)
recent = ps[ps.year >= 2024].groupby("name").size().index
extra = [n for n in recent if n not in set(P.cs_name.dropna())]
P = pd.concat([P, pd.DataFrame({"key": [norm(n) for n in extra], "name": extra, "cs_name": extra})], ignore_index=True)
P["has_stats"] = P.cs_name.notna()

# final role: bowlers split into fast / spin from the overs pattern; stats-only players get a role from their stats
tot = ps.groupby("name")[["balls", "legal_balls"]].sum()
def final_role(r):
    role = r.role_raw
    if role is None and r.cs_name in tot.index:
        b, w = tot.loc[r.cs_name]
        role = "Bowler" if b < 60 and w >= 60 else ("All-rounder" if b >= 150 and w >= 150 else ("Batter" if b >= w else "Bowler"))
    if role == "Bowler":
        return bk.loc[r.cs_name, "kind"] if r.cs_name in bk.index else "Fast bowler"
    return role
P["role"] = P.apply(final_role, axis=1)
P["bowler_kind"] = P.apply(lambda r: bk.loc[r.cs_name, "kind"] if r.cs_name in bk.index else None, axis=1)
OS = """Jos Buttler|Heinrich Klaasen|Ryan Rickelton|Cameron Green|Travis Head|Philip Salt|Phil Salt|Sunil Narine|Lungi Ngidi|Tristan Stubbs|Andre Russell|Marcus Stoinis|Sam Curran|Rashid Khan|Donovan Ferreira|Andrew Tye|Mitchell Starc|Jamie Overton|Will Jacks|Rilee Rossouw|Jake Fraser-McGurk|Anrich Nortje|Azmatullah Omarzai|AM Ghazanfar|Shai Hope|Rachin Ravindra|Nandre Burger|Matthew Short|Kane Williamson|Harry Brook|Spencer Johnson|Xavier Bartlett|Jacob Bethell|Sikandar Raza|Tom Kohler-Cadmore|Kwena Maphaka|Dushmantha Chameera|Matheesha Pathirana|Nuwan Thushara|Adam Zampa|Mujeeb Rahman|Sediqullah Atal|Eshan Malinga|Shamar Joseph|Duan Jansen|Matthew Breetzke|Michael Bracewell|Gulbadin Naib|Ashton Turner""".split("|")
OSK = {norm(x) for x in OS}
def natflag(r):
    n = str(r.nationality).strip().lower() if r.nationality is not None and r.nationality == r.nationality else ""
    if norm(r["name"]) in OSK: return "Overseas"
    if n in ("india", "indian"): return "Indian"
    if n and n != "nan": return "Overseas"
    return None
P["nationality"] = P.apply(natflag, axis=1)
P["nat_source"] = P.nationality.map(lambda x: "file/manual" if x else "unknown")
P.to_csv(f"{OUT}/players_master.csv", index=False)
print(len(P), "people |", P.has_stats.sum(), "with IPL stats")
print(P.role.value_counts(dropna=False).to_dict())
print("nationality known:", P.nationality.notna().sum(), " age known:", P.age_now.notna().sum())
rec = P[P.cs_name.isin(recent)]
print("active in 2024-26:", len(rec), "| nationality known:", rec.nationality.notna().sum(), "| age known:", rec.age_now.notna().sum())
print("active in 2024-26 with no auction row:", len(extra))


In [ ]:
!python -I /content/scripts/step4_players.py

## Step 6. Train and test the price model (train 2021-2024, test 2025)

In [ ]:
%%writefile /content/scripts/step5_model.py
import sys; sys.path.insert(0, "/content/scripts")
import pandas as pd, numpy as np
from common import *
from sklearn.ensemble import HistGradientBoostingRegressor as H
A = pd.read_csv(f"{OUT}/auctions_clean.csv"); M = pd.read_csv(f"{OUT}/name_map.csv")
P = pd.read_csv(f"{OUT}/players_master.csv"); ps = pd.read_csv(f"{OUT}/player_season.csv")
A = A.merge(M[["key","cs_name"]], on="key", how="left").merge(P[["key","role","nationality"]].rename(columns={"role":"role_f","nationality":"nat_f"}), on="key", how="left")

def feats(name, Y):
    d = ps[(ps.name == name) & (ps.year < Y)]
    f = dict(seasons=d.year.nunique(), last_gap=(Y - d.year.max()) if len(d) else np.nan)
    if not len(d): return f
    w = np.where(d.year == Y - 1, 2.0, np.where(d.year == Y - 2, 1.5, 1.0))
    def s(c): return float((d[c] * w).sum())
    b, r, o = s("balls"), s("runs"), s("outs")
    f.update(bat_balls=d.balls.sum(), bat_runs=d.runs.sum(),
        sr=100*r/b if b >= 30 else np.nan, avg=r/max(o,1) if b >= 30 else np.nan,
        six_rate=s("sixes")/b if b >= 30 else np.nan, boundary=(s("fours")+s("sixes"))/b if b >= 30 else np.nan)
    lb = s("legal_balls")
    f.update(bowl_balls=d.legal_balls.sum(), wkts=d.wickets.sum(),
        econ=6*s("conceded")/lb if lb >= 48 else np.nan, wkt_rate=s("wickets")/lb*24 if lb >= 48 else np.nan,
        death_share=d.balls_death.sum()/max(d.legal_balls.sum(),1) if d.legal_balls.sum() >= 48 else np.nan)
    cb, cr_, co = d.balls.sum(), d.runs.sum(), d.outs.sum(); cl = d.legal_balls.sum()
    f.update(c_sr=100*cr_/cb if cb >= 60 else np.nan, c_avg=cr_/max(co,1) if cb >= 60 else np.nan,
             c_econ=6*d.conceded.sum()/cl if cl >= 96 else np.nan, c_wkt=d.wickets.sum()/cl*24 if cl >= 96 else np.nan)
    last = d[d.year == Y - 1]
    f["last_runs"] = last.runs.sum() if len(last) else 0; f["last_wkts"] = last.wickets.sum() if len(last) else 0
    return f

S = A[A.status == "sold"].dropna(subset=["price_cr"]).copy()
F = pd.DataFrame([feats(r.cs_name, r.year) if isinstance(r.cs_name, str) else {} for r in S.itertuples()], index=S.index)
X = pd.concat([S[["year","price_cr","base_cr","age_at_auction","capped","player","role_f","nat_f"]], F], axis=1)
X["overseas"] = np.where(X.nat_f.isna(), np.nan, (X.nat_f == "Overseas").astype(float))
X["capped"] = X.capped.astype(str).str.lower().map({"capped":1.0,"true":1.0,"uncapped":0.0,"false":0.0})
for r in ["Batter","Wicketkeeper","All-rounder","Fast bowler","Spin bowler"]: X["r_"+r] = (X.role_f == r).astype(int)
X["pool_year"] = X.year
cols = [c for c in X.columns if c not in ["year","price_cr","player","role_f","nat_f","base_cr"]]
X["lp"] = np.log(X.price_cr)
tr, te = X[X.year <= 2024], X[X.year == 2025]
print("train", len(tr), "test", len(te))
def run(cs, name):
    m = H(max_depth=3, learning_rate=0.05, max_iter=200, min_samples_leaf=8, l2_regularization=1.0, random_state=0).fit(tr[cs], tr.lp)
    p = np.exp(m.predict(te[cs])); e = (p - te.price_cr).abs()
    print(f"{name:28s} MAE {e.mean():.2f} cr | median {e.median():.2f} | corr {np.corrcoef(p, te.price_cr)[0,1]:.2f}"); return m, p
base1 = np.full(len(te), tr.price_cr.mean()); print("baseline mean price         MAE %.2f" % (te.price_cr-base1).abs().mean())
rm = tr.groupby("r_Batter").price_cr.mean()
rb = tr.assign(role=tr.role_f).groupby("role").price_cr.mean(); pb = te.role_f.map(rb).fillna(tr.price_cr.mean())
print("baseline avg by role        MAE %.2f" % (te.price_cr-pb).abs().mean())
print("baseline = base price       MAE n/a (base missing for many)")
m1, p1 = run(cols, "model (no base price)")
m2, p2 = run(cols + ["base_cr"], "model (with base price)")
te = te.assign(pred=p1); te["err"] = te.pred - te.price_cr
print(te.sort_values("price_cr", ascending=False)[["player","price_cr","pred"]].head(12).round(2).to_string(index=False))
mt = te.copy()
exp = te.seasons.fillna(0)
for name, f in [("none", lambda e: 1.0), ("sqrt(bat+bowl balls/900)", None)]:
    pass
te["exp"] = (te.bat_balls.fillna(0) + te.bowl_balls.fillna(0))
for k in [400, 800, 1200]:
    fac = np.sqrt(np.minimum(1, te.exp / k))
    pp = p1 * fac.values
    print(f"experience shrink k={k}: MAE {(pp - te.price_cr).abs().mean():.2f}")
X.to_pickle(f"{OUT}/model_table.pkl")


In [ ]:
!python -I /content/scripts/step5_model.py

## Step 7. Final model, fair prices and scores, export

In [ ]:
%%writefile /content/scripts/step6_export.py
import sys, json; sys.path.insert(0, "/content/scripts")
import pandas as pd, numpy as np
exec(open("/content/scripts/step5_model.py").read().split("S = A[A.status")[0])  # reuse feats()
X = pd.read_pickle(f"{OUT}/model_table.pkl")
cols = [c for c in X.columns if c not in ["year","price_cr","player","role_f","nat_f","base_cr","lp","pool_year"]]
m = H(max_depth=3, learning_rate=0.05, max_iter=200, min_samples_leaf=8, l2_regularization=1.0, random_state=0).fit(X[cols], X.lp)
AUCTION_YEAR = 2027   # <- change this for the next auction
Y = AUCTION_YEAR
rec = set(ps[ps.year >= 2025].name)
P2 = P[P.cs_name.isin(rec) | (P.last_auction_year >= 2025)].copy()
P2 = P2[P2.role.notna()]
P2["_n"] = P2.cs_name.notna().astype(int) + P2.last_auction_year.fillna(0)/10000
P2 = P2.sort_values("_n", ascending=False)
P2 = pd.concat([P2[P2.cs_name.notna()].drop_duplicates("cs_name"), P2[P2.cs_name.isna()].drop_duplicates("key")])
rows = []
for r in P2.itertuples():
    f = feats(r.cs_name, Y) if isinstance(r.cs_name, str) else {}
    d = dict(f); d["age_at_auction"] = (r.age_now + 1) if pd.notna(r.age_now) else np.nan
    d["capped"] = np.nan
    d["overseas"] = np.nan if pd.isna(r.nationality) else float(r.nationality == "Overseas")
    for ro in ["Batter","Wicketkeeper","All-rounder","Fast bowler","Spin bowler"]: d["r_"+ro] = int(r.role == ro)
    rows.append(d)
D = pd.DataFrame(rows).reindex(columns=cols)
P2["fair_cr"] = np.exp(m.predict(D)).round(2)
P2["has_stats"] = P2.cs_name.notna() & D["seasons"].fillna(0).values.astype(float).__gt__(0)
P2["fair_cr"] = np.where(P2.has_stats, P2.fair_cr, 0.30)
last = ps[ps.year == 2026].set_index("name"); l25 = ps[ps.year == 2025].set_index("name")
def cur_team(n):
    for t in (last, l25):
        if n in t.index and isinstance(t.loc[n,"team"], str): return t.loc[n,"team"]
    return None
P2["team_now"] = P2.cs_name.map(lambda n: cur_team(n) if isinstance(n,str) else None)
P2["team_now"] = P2.team_now.fillna(P2.last_team.map(lambda t: team_code(t) if isinstance(t,str) else None))
# score 0-100 = rank of fair price among players with stats
hs = P2.has_stats
P2["score"] = 30.0; P2.loc[hs, "score"] = (P2.loc[hs, "fair_cr"].rank(pct=True) * 100).round(0)
# plain-words stats line
def line(r):
    f = feats(r.cs_name, Y) if isinstance(r.cs_name, str) else {}
    out = []
    if f.get("bat_balls", 0) >= 60 and pd.notna(f.get("sr")): out.append(f"SR {f['sr']:.0f}, avg {f['avg']:.0f}")
    if f.get("bowl_balls", 0) >= 60 and pd.notna(f.get("econ")): out.append(f"econ {f['econ']:.1f}, {int(f['wkts'])} wkts")
    return " | ".join(out) if out else "No IPL stats yet"
P2["stats"] = P2.apply(line, axis=1)
def expb(r):
    f = feats(r.cs_name, Y) if isinstance(r.cs_name, str) else {}
    return float(f.get("bat_balls", 0) or 0) + float(f.get("bowl_balls", 0) or 0)
P2["exp_balls"] = P2.apply(expb, axis=1)
P2["base_cr"] = P2.last_base_cr.fillna(0.30); P2["base_assumed"] = P2.last_base_cr.isna()
out = P2[["name","role","nationality","age_now","team_now","score","fair_cr","base_cr","base_assumed","stats","has_stats","last_auction_year","exp_balls"]].copy()
out["age_now"] = out.age_now.astype("Int64").astype(object).where(out.age_now.notna(), None)
out["nationality"] = out.nationality.where(out.nationality.notna(), None)
out["team_now"] = out.team_now.where(out.team_now.notna(), None)
out = out.sort_values("fair_cr", ascending=False)
out.to_csv(f"{OUT}/players_pool.csv", index=False)
print(len(out), "players;", out.has_stats.sum(), "with stats;", out.nationality.notna().sum(), "nationality known;", out.age_now.notna().sum(), "age known")
print(out.head(12)[["name","role","team_now","fair_cr","score","stats"]].to_string(index=False))
# teams
T = pd.read_csv(f"{OUT}/team_season.csv"); t = T[T.year == 2026].copy()
t["bat_rank"] = t.bat_rpo.rank(ascending=False); t["bowl_rank"] = t.bowl_rpo.rank(ascending=True)
print(t[["team","pos","W","L","bat_rpo","bowl_rpo","bat_rank","bowl_rank"]].sort_values("pos").to_string(index=False))
t.to_csv(f"{OUT}/teams_2026.csv", index=False)


In [ ]:
!python -I /content/scripts/step6_export.py

## Download the results
These two files feed the app.

In [ ]:
files.download('/content/out/players_pool.csv'); files.download('/content/out/teams_2026.csv')